# Task 3 - CPU Deployment of the Story AI System

This notebook contains the final CPU deployment work for the Task 1 chatbot. It records the rejected Qwen2.5-7B baseline, a controlled 7B/3B/1.5B GGUF model-size ablation, and the selected Qwen2.5-1.5B Q4_K_M deployment using compact query-relevant evidence chunks.

The final execution path loads only the selected 1.5B model. Rejected 7B and 3B conversion logs were removed from the runnable path; their measured benchmark results are retained as evidence. Task 2 classifier deployment is documented separately in the accompanying Task 3 documentation.


In [ ]:
# CELL 1 — Mount Drive (separate session, needs its own mount)
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [1]:
TASK3_DIR = "/content/drive/MyDrive/task3_cpu_deployment"


In [5]:
!pip install -q llama-cpp-python

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.9/74.9 MB 8.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.4 MB/s eta 0:00:00


In [12]:
# CELL 2 — Clone llama.cpp + build it (this is the slow part — let it run)
!git clone --depth 1 https://github.com/ggml-org/llama.cpp.git /content/llama.cpp
!pip install -q -r /content/llama.cpp/requirements/requirements-convert_hf_to_gguf.txt
!cmake -B /content/llama.cpp/build -S /content/llama.cpp -DCMAKE_BUILD_TYPE=Release
!cmake --build /content/llama.cpp/build --config Release -j $(nproc) --target llama-quantize


Cloning into '/content/llama.cpp'...
remote: Enumerating objects: 3859, done.
remote: Counting objects: 100% (3859/3859), done.
remote: Compressing objects: 100% (3149/3149), done.
remote: Total 3859 (delta 688), reused 2675 (delta 628), pack-reused 0 (from 0)
Receiving objects: 100% (3859/3859), 35.57 MiB | 18.59 MiB/s, done.
Resolving deltas: 100% (688/688), done.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 90.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 71.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.5/118.5 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/56

In [21]:
%cd /content/llama.cpp

!cmake -B build
!cmake --build build --config Release -j 2

/content/llama.cpp
-- llama.cpp version: 0.3.0-dev
CMAKE_BUILD_TYPE=Release
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Including CPU backend
-- x86 detected
-- Adding CPU backend variant ggml-cpu: -march=native 
-- ggml version: 0.22.0
-- ggml commit:  daef7b6
-- OpenSSL found: 3.0.2
-- Generating embedded license file for target: llama-app
-- Configuring done (0.5s)
-- Generating done (0.6s)
-- Build files have been written to: /content/llama.cpp/build
[  0%] Building CXX object vendor/hash/CMakeFiles/vendor-hash.dir/hash.cpp.o
[  2%] Built target ggml-base
[  2%] Built target cpp-httplib
[  2%] Building C object vendor/hash/CMakeFiles/vendor-hash.dir/xxhash/xxhash.c.o
[  2%] Built target llama-common-base
[  2%] Building CXX object vendor/hash/CMakeFiles/vendor-hash.dir/sha1/sha1.c.o
[  3%] Building CXX object tools/ui/CMakeFiles/llama-ui-embed.

In [22]:
!ls /content/llama.cpp/build/bin | grep llama

libllama-batched-bench-impl.so
libllama-bench-impl.so
libllama-cli-impl.so
libllama-common.so
libllama-common.so.0
libllama-common.so.0.3.0
libllama-completion-impl.so
libllama-fit-params-impl.so
libllama-perplexity-impl.so
libllama-quantize-impl.so
libllama-server-impl.so
libllama.so
libllama.so.0
libllama.so.0.3.0
llama
llama-batched
llama-batched-bench
llama-bench
llama-cli
llama-completion
llama-convert-llama2c-to-ggml
llama-cvector-generator
llama-debug
llama-diffusion-cli
llama-embedding
llama-eval-callback
llama-export-lora
llama-finetune
llama-fit-params
llama-gemma3-cli
llama-gen-docs
llama-gguf
llama-gguf-hash
llama-gguf-split
llama-idle
llama-imatrix
llama-llava-cli
llama-lookahead
llama-lookup
llama-lookup-create
llama-lookup-merge
llama-lookup-stats
llama-minicpmv-cli
llama-mtmd-cli
llama-mtmd-debug
llama-parallel
llama-passkey
llama-perplexity
llama-q8dot
llama-quantize
llama-qwen2vl-cli
llama-results
llama-retrieval
llama-server
llama-simple
llama-simple-chat
llama-specu

In [ ]:
import os
print(f"CPU count: {os.cpu_count()}")
!nproc
!cat /proc/cpuinfo | grep processor | wc -l

CPU count: 2
2
2


## Use smaller model(qwe.2.5_1.5b)

In [8]:
SMALL_HF_DIR = "/content/qwen2.5-1.5b-hf"
SMALL_F16_GGUF = "/content/qwen2.5-1.5b-instruct-f16.gguf"
SMALL_Q4_GGUF = "/content/qwen2.5-1.5b-instruct-q4_k_m.gguf"

In [10]:
!hf download Qwen/Qwen2.5-1.5B-Instruct \
    --local-dir /content/qwen2.5-1.5b-hf

Hint: The `hf-cli` skill is not installed. Run `hf skills add -g --claude` to teach your AI agents how to use the `hf` CLI.
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0% 0/10 [00:00<?, ?it/s]Still waiting to acquire lock on /content/qwen2.5-1.5b-hf/.cache/huggingface/.gitignore.lock (elapsed: 0.1 seconds)

Reconstructing (incomplete total...):   0% 0.00/3.09G [00:00<?, ?B/s]         
Reconstructing (incomplete total...):   0% 0.00/3.09G [00:00<?, ?B/s]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.

Reconstructing (incomplete total...):   0% 0.00/3.09G [00:00<?, ?B/s]

Fetching 10 files:  10% 1/10 [00:00<00:02,  3.41it/s]
Reconstructing (incomplete total...):   0% 11.3k/3.09G [00:00<21:52:12, 39.3kB/s]
Reconstructing (incomplete total...):   0% 12.0k/3.09G [00:00<21:52:12, 39.3kB/s]
Reconstructing (incomplete total...):   0% 7.05M/3.09G [00

In [13]:
!python /content/llama.cpp/convert_hf_to_gguf.py \
    /content/qwen2.5-1.5b-hf \
    --outfile /content/qwen2.5-1.5b-instruct-f16.gguf \
    --outtype f16

INFO:hf-to-gguf:Loading model: qwen2.5-1.5b-hf
INFO:hf-to-gguf:Model architecture: Qwen2ForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:token_embd.weight,         torch.bfloat16 --> F16, shape = {1536, 151936}
INFO:hf-to-gguf:blk.0.attn_norm.weight,    torch.bfloat16 --> F32, shape = {1536}
INFO:hf-to-gguf:blk.0.ffn_down.weight,     torch.bfloat16 --> F16, shape = {8960, 1536}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,     torch.bfloat16 --> F16, shape = {1536, 8960}
INFO:hf-to-gguf:blk.0.ffn_up.weight,       torch.bfloat16 --> F16, shape = {1536, 8960}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,     torch.bfloat16 --> F32, shape = {1536}
INFO:hf-to-gguf:blk.0.attn_k.bias,         torch.bfloat16 --> F32, shape = {256}
INFO:hf-to-gguf:blk.0.attn_k.weight,       torch.bfloat16 --> F16, shape = {1536, 256}
INFO:hf-to-gguf:blk.0.attn_output.weight,  torch

In [14]:
import os

f16_size = os.path.getsize(SMALL_F16_GGUF) / 1024**3
print(f"FP16 GGUF: {f16_size:.2f} GiB")

FP16 GGUF: 2.88 GiB


In [15]:
!/content/llama.cpp/build/bin/llama-quantize \
    /content/qwen2.5-1.5b-instruct-f16.gguf \
    /content/qwen2.5-1.5b-instruct-q4_k_m.gguf \
    Q4_K_M

!cp /content/qwen2.5-1.5b-instruct-q4_k_m.gguf {TASK3_DIR}/gguf/

version: 0.3.0-dev (build 1, commit daef7b6)
built with GNU 11.4.0 for Linux x86_64
llama_quantize: quantizing '/content/qwen2.5-1.5b-instruct-f16.gguf' to '/content/qwen2.5-1.5b-instruct-q4_k_m.gguf' as Q4_K_M
llama_model_loader: loaded meta data with 38 key-value pairs and 338 tensors from /content/qwen2.5-1.5b-instruct-f16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = qwen2
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                     general.sampling.top_k i32              = 20
llama_model_loader: - kv   3:                     general.sampling.top_p f32              = 0.800000
llama_model_loader: - kv   4:                      general.sampling.temp f32              = 0.700000
llama_model_loader: - kv   5:            general.

In [16]:
q4_size = os.path.getsize(SMALL_Q4_GGUF) / 1024**3
print(f"Q4_K_M GGUF: {q4_size:.2f} GiB")

Q4_K_M GGUF: 0.92 GiB


In [17]:
import os

os.remove(SMALL_F16_GGUF)
print("Removed temporary FP16 GGUF")

Removed temporary FP16 GGUF


In [23]:
!/content/llama.cpp/build/bin/llama-bench \
    -m /content/qwen2.5-1.5b-instruct-q4_k_m.gguf \
    -p 128 -n 128 -t $(nproc)

| model                          |       size |     params | backend    | threads |            test |                  t/s |
| ------------------------------ | ---------: | ---------: | ---------- | ------: | --------------: | -------------------: |
| qwen2 1.5B Q4_K - Medium       | 934.69 MiB |     1.54 B | CPU        |       2 |           pp128 |         27.95 ± 3.53 |
| qwen2 1.5B Q4_K - Medium       | 934.69 MiB |     1.54 B | CPU        |       2 |           tg128 |         11.43 ± 1.82 |

build: daef7b6 (1)


In [32]:
import os
import json

BENCHMARK_DIR = f"{TASK3_DIR}/benchmarks"

os.makedirs(BENCHMARK_DIR, exist_ok=True)

model_size_ablation = {
    "hardware": {
        "cpu": "Intel Xeon @ 2.20GHz",
        "logical_cpus": 2,
        "threads_used": 2,
    },
    "benchmark": {
        "tool": "llama-bench",
        "prompt_test": "pp128",
        "generation_test": "tg128",
    },
    "models": [
        {
            "model": "Qwen2.5-7B-Instruct",
            "quantization": "GGUF Q4_K_M",
            "size_gib": 4.36,
            "parameters_billion": 7.62,
            "prompt_processing_tok_s": 3.26,
            "generation_tok_s": 1.53,
            "decision": "rejected_due_to_extreme_latency",
        },
        {
            "model": "Qwen2.5-3B-Instruct",
            "quantization": "GGUF Q4_K_M",
            "size_gib": 1.79,
            "parameters_billion": 3.09,
            "prompt_processing_tok_s": 13.49,
            "generation_tok_s": 6.02,
            "decision": "not_selected_due_to_latency",
        },
        {
            "model": "Qwen2.5-1.5B-Instruct",
            "quantization": "GGUF Q4_K_M",
            "size_mib": 934.69,
            "parameters_billion": 1.54,
            "prompt_processing_tok_s": 27.95,
            "generation_tok_s": 11.43,
            "decision": "selected_for_end_to_end_rag_evaluation",
        },
    ],
}

ablation_path = (
    f"{BENCHMARK_DIR}/"
    "task1_cpu_model_size_ablation.json"
)

with open(ablation_path, "w") as file:
    json.dump(
        model_size_ablation,
        file,
        indent=2,
    )

print("Saved:", ablation_path)

Saved: /content/drive/MyDrive/task3_cpu_deployment/benchmarks/task1_cpu_model_size_ablation.json


## CPU model-size ablation

All models used GGUF Q4_K_M and were benchmarked with two CPU threads.

| Model | Size | Prompt processing | Generation | Decision |
|---|---:|---:|---:|---|
| Qwen2.5-7B | 4.36 GiB | 3.26 tok/s | 1.53 tok/s | Rejected |
| Qwen2.5-3B | 1.79 GiB | 13.49 tok/s | 6.02 tok/s | Not selected |
| Qwen2.5-1.5B | 934.69 MiB | 27.95 tok/s | 11.43 tok/s | Selected |

The 1.5B model was selected because it was approximately twice as fast
as the 3B model and more than seven times faster than the 7B model while
requiring less than 1 GiB for the GGUF artifact.

## Rejected 7B full-context baseline

Qwen2.5-7B-Instruct Q4_K_M was initially evaluated using complete
story contexts on the two-core CPU target.

| Query | Prompt tokens | TTFT | Total latency |
|---|---:|---:|---:|
| Exact ID query | 1,303 | 498.5 s | 637.7 s |
| Semantic query | 2,893 | 1,043.2 s | 1,152.8 s |

The complete RAG process used approximately 6.69 GiB RSS. Retrieval
required only 0.3–181.5 ms, showing that LLM prompt processing and
generation were the primary bottlenecks.

This configuration was rejected because it could execute on CPU but
did not meet the practical low-latency objective. Blind 300-character
and head-tail context reductions were also rejected because their sample
answers contained unsupported or generic content.

In [33]:
import os
import gc
import time
import json
import re
import platform

import numpy as np
import psutil

os.environ["OMP_NUM_THREADS"] = "2"
os.environ["MKL_NUM_THREADS"] = "2"

SMALL_Q4_GGUF = (
    "/content/qwen2.5-1.5b-instruct-q4_k_m.gguf"
)

BENCHMARK_DIR = f"{TASK3_DIR}/benchmarks"
os.makedirs(BENCHMARK_DIR, exist_ok=True)

assert os.path.exists(SMALL_Q4_GGUF), (
    f"Model not found: {SMALL_Q4_GGUF}"
)

process = psutil.Process(os.getpid())

print("CPU logical cores:", psutil.cpu_count(logical=True))
print("CPU physical cores:", psutil.cpu_count(logical=False))
print("Model path:", SMALL_Q4_GGUF)
print(
    "Model size:",
    f"{os.path.getsize(SMALL_Q4_GGUF) / 1024**2:.2f} MiB",
)

CPU logical cores: 2
CPU physical cores: 1
Model path: /content/qwen2.5-1.5b-instruct-q4_k_m.gguf
Model size: 940.37 MiB


In [35]:
from llama_cpp import Llama

load_start = time.perf_counter()

llm_small = Llama(
    model_path=SMALL_Q4_GGUF,
    n_ctx=2048,
    n_threads=2,
    n_threads_batch=2,
    n_batch=256,
    use_mmap=True,
    use_mlock=False,
    verbose=False,
)

small_model_load_s = time.perf_counter() - load_start
rss_after_model_load_gib = (
    process.memory_info().rss / 1024**3
)

print("Qwen2.5-1.5B Q4_K_M loaded")
print(f"Load time: {small_model_load_s:.2f}s")
print(f"RSS after model load: {rss_after_model_load_gib:.2f} GiB")

Qwen2.5-1.5B Q4_K_M loaded
Load time: 7.35s
RSS after model load: 2.42 GiB


## Smoke Test for the model

In [36]:
llm_small.reset()

start = time.perf_counter()

output = llm_small.create_chat_completion(
    messages=[
        {
            "role": "user",
            "content": "What is the capital of France? Answer briefly.",
        }
    ],
    temperature=0,
    max_tokens=20,
)

simple_test_time = time.perf_counter() - start
simple_answer = output["choices"][0]["message"]["content"]

print("Answer:", simple_answer)
print(f"Time: {simple_test_time:.2f}s")

Answer: The capital of France is Paris.
Time: 2.83s


In [2]:
# ============================================================
# CELL 1  — RAG infrastructure with hybrid retrieval
# Requires: TASK3_DIR, GGUF_PATH, llm already defined/loaded
# ============================================================
import os
os.environ["USE_TF"] = "0"

!pip install -q faiss-cpu sentence-transformers rank_bm25

import requests, csv, io, re
import psutil
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi

os.makedirs(f"{TASK3_DIR}/benchmarks", exist_ok=True)

url = "https://huggingface.co/datasets/FareedKhan/1k_stories_100_genre/resolve/main/1k_stories_100_genre.csv"
csv_text = requests.get(url).text
stories = list(csv.DictReader(io.StringIO(csv_text)))
print(f"Loaded {len(stories)} stories")

id_to_row = {int(row['id']): i for i, row in enumerate(stories)}
title_to_rows = {}
genre_to_rows = {}
for i, row in enumerate(stories):
    title_to_rows.setdefault(row['title'].strip().lower(), []).append(i)
    genre_to_rows.setdefault(row['genre'].strip().lower(), []).append(i)
print(f"IDs: {len(id_to_row)}, titles: {len(title_to_rows)}, genres: {len(genre_to_rows)}")

embed_model = SentenceTransformer("BAAI/bge-small-en-v1.5", device="cpu")

# --- Hybrid retrieval infrastructure: chunk-level dense + BM25 ---
def chunk_story(text, chunk_size=500, overlap=100):
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunks.append(' '.join(words[start:end]))
        start += chunk_size - overlap
        if end >= len(words):
            break
    return chunks

chunk_records = []
for row in stories:
    for i, chunk_text in enumerate(chunk_story(row['story'])):
        chunk_records.append({
            'story_id': int(row['id']),  # FIX: cast to int here — CSV gives strings,
                                          # but id_to_row keys are int. Without this,
                                          # every hybrid result was silently filtered out.
            'title': row['title'], 'genre': row['genre'],
            'chunk_index': i, 'chunk_text': chunk_text,
        })
print(f"Built {len(chunk_records)} chunks from {len(stories)} stories "
      f"(avg {len(chunk_records)/len(stories):.1f} chunks/story)")

chunk_texts = [c['chunk_text'] for c in chunk_records]
print("Encoding chunks...")
chunk_embeddings = embed_model.encode(chunk_texts, batch_size=32, show_progress_bar=True,
                                        normalize_embeddings=True)
chunk_embeddings = np.array(chunk_embeddings).astype('float32')

chunk_index = faiss.IndexFlatIP(chunk_embeddings.shape[1])
chunk_index.add(chunk_embeddings)
print(f"Chunk FAISS index: {chunk_index.ntotal} vectors")

tokenized_chunks = [c.lower().split() for c in chunk_texts]
bm25_index = BM25Okapi(tokenized_chunks)
print("BM25 index built")

process = psutil.Process(os.getpid())
rss_after_rag_load_mb = process.memory_info().rss / 1024**2
print(f"Full CPU system RSS after loading Qwen + embedder + hybrid indexes: {rss_after_rag_load_mb:.0f} MB")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 94.7 MB/s eta 0:00:00
Loaded 1000 stories
IDs: 1000, titles: 998, genres: 99


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Built 2764 chunks from 1000 stories (avg 2.8 chunks/story)
Encoding chunks...


Batches:   0%|          | 0/87 [00:00<?, ?it/s]

Chunk FAISS index: 2764 vectors
BM25 index built
Full CPU system RSS after loading Qwen + embedder + hybrid indexes: 1263 MB
Infrastructure + hybrid retrieval ready (story_id type fix applied)


In [37]:
required_variables = [
    "stories",
    "id_to_row",
    "title_to_rows",
    "genre_to_rows",
    "chunk_records",
    "chunk_embeddings",
    "chunk_index",
    "bm25_index",
    "embed_model",
]

for variable in required_variables:
    print(variable, "available:", variable in globals())

stories available: True
id_to_row available: True
title_to_rows available: True
genre_to_rows available: True
chunk_records available: True
chunk_embeddings available: True
chunk_index available: True
bm25_index available: True
embed_model available: True


In [38]:
def retrieve_hybrid_evidence(
    query,
    top_k_chunks=20,
    top_k_stories=3,
    rrf_k=60,
):
    query_embedding = embed_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype("float32")

    _, dense_indices = chunk_index.search(
        query_embedding,
        top_k_chunks,
    )

    dense_ranked = [
        int(index)
        for index in dense_indices[0]
        if int(index) >= 0
    ]

    bm25_scores = bm25_index.get_scores(
        query.lower().split()
    )

    bm25_ranked = [
        int(index)
        for index in np.argsort(bm25_scores)[::-1][
            :top_k_chunks
        ]
    ]

    fused_scores = {}

    for rank, chunk_idx in enumerate(
        dense_ranked,
        start=1,
    ):
        fused_scores[chunk_idx] = (
            fused_scores.get(chunk_idx, 0.0)
            + 1.0 / (rrf_k + rank)
        )

    for rank, chunk_idx in enumerate(
        bm25_ranked,
        start=1,
    ):
        fused_scores[chunk_idx] = (
            fused_scores.get(chunk_idx, 0.0)
            + 1.0 / (rrf_k + rank)
        )

    # Keep only the best chunk for each distinct story.
    best_by_story = {}

    for chunk_idx, fused_score in fused_scores.items():
        record = chunk_records[chunk_idx]
        story_id = int(record["story_id"])

        previous = best_by_story.get(story_id)

        if (
            previous is None
            or fused_score > previous["score"]
        ):
            best_by_story[story_id] = {
                "story_id": story_id,
                "title": record["title"],
                "genre": record["genre"],
                "chunk_index": int(
                    record.get("chunk_index", 0)
                ),
                "chunk_text": record["chunk_text"],
                "score": float(fused_score),
            }

    ranked_evidence = sorted(
        best_by_story.values(),
        key=lambda item: item["score"],
        reverse=True,
    )

    return ranked_evidence[:top_k_stories]

In [39]:
test_query = (
    "Find me a story about adventure and friendship"
)

retrieval_start = time.perf_counter()

test_evidence = retrieve_hybrid_evidence(
    test_query,
    top_k_stories=3,
)

retrieval_ms = (
    time.perf_counter() - retrieval_start
) * 1000

print(f"Retrieval time: {retrieval_ms:.1f} ms")

for rank, item in enumerate(test_evidence, start=1):
    print(
        f"{rank}. ID={item['story_id']} | "
        f"Title={item['title']} | "
        f"Chunk={item['chunk_index']} | "
        f"RRF={item['score']:.6f}"
    )

Retrieval time: 1139.1 ms
1. ID=738701 | Title=A Summer to Remember | Chunk=0 | RRF=0.029139
2. ID=833361 | Title=The Unlikely Hero of Crescent Bay | Chunk=2 | RRF=0.027242
3. ID=869376 | Title=The Unyielding Soul | Chunk=0 | RRF=0.016393


In [40]:
def retrieve_evidence_within_story(
    query,
    story_id,
    top_n=1,
):
    story_id = int(story_id)

    candidate_indices = [
        idx
        for idx, record in enumerate(chunk_records)
        if int(record["story_id"]) == story_id
    ]

    if not candidate_indices:
        return []

    query_embedding = embed_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype("float32")[0]

    candidate_matrix = chunk_embeddings[
        candidate_indices
    ]

    dense_scores = (
        candidate_matrix @ query_embedding
    )

    ranked_positions = np.argsort(
        dense_scores
    )[::-1][:top_n]

    results = []

    for position in ranked_positions:
        candidate_position = int(position)
        chunk_idx = candidate_indices[
            candidate_position
        ]
        record = chunk_records[chunk_idx]

        results.append({
            "story_id": story_id,
            "title": record["title"],
            "genre": record["genre"],
            "chunk_index": int(
                record.get("chunk_index", 0)
            ),
            "chunk_text": record["chunk_text"],
            "score": float(
                dense_scores[candidate_position]
            ),
        })

    return results

In [41]:
id_evidence = retrieve_evidence_within_story(
    query=(
        "What supernatural force threatens "
        "the family in story ID 523790?"
    ),
    story_id=523790,
    top_n=1,
)

for item in id_evidence:
    print(item["story_id"], item["title"])
    print(item["chunk_text"][:500])

523790 Willowbrook Manor's Ghostly Echoes
determined than ever, set out to find the tome. They scoured the manor from top to bottom, searching every nook and cranny. They encountered the Whispering Trees in various forms, from ghostly apparitions to twisted, gnarled tree branches that seemed to come to life. With each encounter, the family's resolve grew stronger, and they became more adept at combating the evil force. Chapter 3: The Race Against Time As the Winstons continued their quest, they discovered that the Whispering Trees were 


In [42]:
def trim_to_words(text, max_words=350):
    words = text.split()

    if len(words) <= max_words:
        return text

    return " ".join(words[:max_words]) + " [...]"


def build_compact_prompt(
    query,
    evidence,
    max_evidence=1,
    max_words_per_chunk=350,
):
    selected = evidence[:max_evidence]

    context_parts = []

    for position, item in enumerate(
        selected,
        start=1,
    ):
        relevant_text = trim_to_words(
            item["chunk_text"],
            max_words=max_words_per_chunk,
        )

        context_parts.append(
            f"STORY {position}\n"
            f"ID: {item['story_id']}\n"
            f"Title: {item['title']}\n"
            f"Genre: {item['genre']}\n"
            f"Relevant story text:\n"
            f"{relevant_text}"
        )

    context = "\n\n---\n\n".join(context_parts)

    return f"""Answer the question using only the supplied story text.

Rules:
- Do not invent characters, events, dialogue, or explanations.
- If the supplied text is insufficient, say that clearly.
- Keep different stories separate.
- Keep the answer concise.

STORY CONTEXT:
{context}

USER QUESTION:
{query}

ANSWER:"""

In [43]:
compact_prompt = build_compact_prompt(
    test_query,
    test_evidence,
    max_evidence=1,
    max_words_per_chunk=350,
)

raw_prompt_tokens = len(
    llm_small.tokenize(
        compact_prompt.encode("utf-8"),
        add_bos=True,
    )
)

# Allow extra space for the Qwen chat template.
estimated_chat_tokens = raw_prompt_tokens + 64

print("Raw prompt tokens:", raw_prompt_tokens)
print(
    "Estimated tokens with chat template:",
    estimated_chat_tokens,
)

Raw prompt tokens: 508
Estimated tokens with chat template: 572


In [44]:
def benchmark_llm_once(
    prompt,
    max_tokens=96,
):
    # Clear previous KV-cache state.
    llm_small.reset()

    start = time.perf_counter()
    first_token_s = None
    answer_parts = []

    for chunk in llm_small.create_chat_completion(
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ],
        temperature=0,
        max_tokens=max_tokens,
        stream=True,
    ):
        delta = (
            chunk["choices"][0]
            .get("delta", {})
            .get("content", "")
        )

        if (
            first_token_s is None
            and delta.strip()
        ):
            first_token_s = (
                time.perf_counter() - start
            )

        answer_parts.append(delta)

    total_s = time.perf_counter() - start
    answer = "".join(answer_parts).strip()

    completion_tokens = len(
        llm_small.tokenize(
            answer.encode("utf-8"),
            add_bos=False,
        )
    )

    return {
        "ttft_s": float(first_token_s),
        "total_s": float(total_s),
        "completion_tokens": completion_tokens,
        "answer": answer,
    }

In [45]:
smoke_result = benchmark_llm_once(
    compact_prompt,
    max_tokens=96,
)

print(f"TTFT: {smoke_result['ttft_s']:.2f}s")
print(f"Total: {smoke_result['total_s']:.2f}s")
print(
    "Completion tokens:",
    smoke_result["completion_tokens"],
)
print("\nANSWER:")
print(smoke_result["answer"])

TTFT: 22.57s
Total: 30.15s
Completion tokens: 90

ANSWER:
A story about adventure and friendship could be about Jack and Lily's summer vacation adventure. They planned a grand adventure to find a hidden treasure in the woods surrounding Willowbrook. Along the way, they faced various challenges and obstacles, but their friendship and determination kept them going. They encountered friendly locals, solved puzzles, and even faced a few dangerous situations. In the end, they found the treasure and returned home with a newfound sense of adventure and friendship.


In [46]:
def find_explicit_story_id(query):
    match = re.search(
        r"\bid\s*:?\s*(\d+)\b",
        query,
        flags=re.IGNORECASE,
    )

    return int(match.group(1)) if match else None


def find_title_rows(query):
    query_lower = query.lower()

    matches = []

    for normalized_title, row_indices in (
        title_to_rows.items()
    ):
        if normalized_title in query_lower:
            matches.extend(row_indices)

    return matches


def find_genre(query):
    query_lower = query.lower().strip()

    patterns = [
        r"\bin the (.+?) genre\b",
        r"\bstories? (?:in|of|from) "
        r"(.+?)(?:\s+genre)?$",
        r"\bgenre[:\s]+(.+)$",
    ]

    for pattern in patterns:
        match = re.search(pattern, query_lower)

        if not match:
            continue

        candidate = match.group(1).strip()

        for genre in genre_to_rows:
            if (
                genre.lower() == candidate
                or genre.lower() in candidate
            ):
                return genre

    return None

In [47]:
def deterministic_genre_response(genre):
    row_indices = genre_to_rows[genre]

    sources = [
        {
            "id": int(stories[idx]["id"]),
            "title": stories[idx]["title"],
            "genre": stories[idx]["genre"],
        }
        for idx in row_indices
    ]

    answer = "\n".join(
        f"- ID {source['id']}: {source['title']}"
        for source in sources
    )

    return {
        "method": "genre_lookup",
        "answer": answer,
        "sources": sources,
        "retrieval_ms": 0.0,
        "ttft_s": 0.0,
        "total_s": 0.0,
        "llm_used": False,
    }

## Final CHATBOT


In [48]:
def cpu_chatbot_query(
    query,
    max_tokens=96,
):
    request_start = time.perf_counter()

    # 1. Exact ID route
    story_id = find_explicit_story_id(query)

    if story_id is not None:
        if story_id not in id_to_row:
            return {
                "method": "id_not_found",
                "answer": (
                    f"No story with ID {story_id} "
                    "was found."
                ),
                "sources": [],
                "retrieval_ms": (
                    time.perf_counter()
                    - request_start
                ) * 1000,
                "llm_used": False,
            }

        evidence = retrieve_evidence_within_story(
            query,
            story_id=story_id,
            top_n=1,
        )

        method = "id_lookup"

    # 2. Exact title route
    else:
        title_rows = find_title_rows(query)

        if title_rows:
            evidence = []

            for row_idx in title_rows:
                title_story_id = int(
                    stories[row_idx]["id"]
                )

                evidence.extend(
                    retrieve_evidence_within_story(
                        query,
                        story_id=title_story_id,
                        top_n=1,
                    )
                )

            method = "title_lookup"

        else:
            # Title routing must precede genre routing.
            genre = find_genre(query)

            if genre is not None:
                result = deterministic_genre_response(
                    genre
                )

                result["retrieval_ms"] = (
                    time.perf_counter()
                    - request_start
                ) * 1000

                return result

            # 3. Hybrid semantic route
            evidence = retrieve_hybrid_evidence(
                query,
                top_k_stories=3,
            )

            method = "semantic_search_hybrid"

    retrieval_ms = (
        time.perf_counter() - request_start
    ) * 1000

    if not evidence:
        return {
            "method": method,
            "answer": (
                "No relevant story evidence "
                "was found."
            ),
            "sources": [],
            "retrieval_ms": retrieval_ms,
            "llm_used": False,
        }

    prompt = build_compact_prompt(
        query,
        evidence,
        max_evidence=1,
        max_words_per_chunk=350,
    )

    prompt_tokens = (
        len(
            llm_small.tokenize(
                prompt.encode("utf-8"),
                add_bos=True,
            )
        )
        + 64
    )

    if prompt_tokens + max_tokens > 2048:
        raise ValueError(
            f"Context limit exceeded: "
            f"{prompt_tokens} prompt tokens + "
            f"{max_tokens} output tokens"
        )

    generation = benchmark_llm_once(
        prompt,
        max_tokens=max_tokens,
    )

    sources = [
        {
            "id": item["story_id"],
            "title": item["title"],
            "genre": item["genre"],
            "chunk_index": item["chunk_index"],
        }
        for item in evidence[:3]
    ]

    return {
        "method": method,
        "llm_used": True,
        "retrieval_ms": float(retrieval_ms),
        "prompt_tokens": int(prompt_tokens),
        "completion_tokens": int(
            generation["completion_tokens"]
        ),
        "ttft_s": generation["ttft_s"],
        "generation_total_s": generation["total_s"],
        "end_to_end_s": float(
            retrieval_ms / 1000
            + generation["total_s"]
        ),
        "answer": generation["answer"],
        "sources": sources,
    }

## TEST

In [49]:
deployment_queries = [
    "What supernatural force threatens the family in story ID 523790?",
    "Find me a story about adventure and friendship.",
    "Which story involves the SS Excelsior and an emergency cloaking device?",
]

deployment_results = []

for query in deployment_queries:
    print("=" * 80)
    print("QUERY:", query)

    result = cpu_chatbot_query(
        query,
        max_tokens=96,
    )

    deployment_results.append({
        "query": query,
        **result,
    })

    print("Method:", result["method"])
    print(
        "Retrieval:",
        f"{result['retrieval_ms']:.1f} ms",
    )

    if result.get("llm_used"):
        print(
            "Prompt tokens:",
            result["prompt_tokens"],
        )
        print(
            "TTFT:",
            f"{result['ttft_s']:.2f}s",
        )
        print(
            "Total:",
            f"{result['end_to_end_s']:.2f}s",
        )

    print("Sources:", result["sources"])
    print("Answer:", result["answer"])

QUERY: What supernatural force threatens the family in story ID 523790?
Method: id_lookup
Retrieval: 236.2 ms
Prompt tokens: 611
TTFT: 22.86s
Total: 24.82s
Sources: [{'id': 523790, 'title': "Willowbrook Manor's Ghostly Echoes", 'genre': 'Horror', 'chunk_index': 1}]
Answer: The supernatural force threatening the family in Story ID 523790 is the Whispering Trees.
QUERY: Find me a story about adventure and friendship.
Method: semantic_search_hybrid
Retrieval: 85.9 ms
Prompt tokens: 426
TTFT: 15.14s
Total: 17.08s
Sources: [{'id': 841055, 'title': 'The Perilous Journey of the Lost Hikers', 'genre': 'Survival', 'chunk_index': 2}, {'id': 738701, 'title': 'A Summer to Remember', 'genre': 'Coming-of-Age', 'chunk_index': 0}, {'id': 103830, 'title': 'The Path to Destiny', 'genre': 'Coming-of-Age', 'chunk_index': 0}]
Answer: The story of the Lost Hikers.
QUERY: Which story involves the SS Excelsior and an emergency cloaking device?
Method: semantic_search_hybrid
Retrieval: 91.5 ms
Prompt tokens: 6

In [50]:
genre_result = cpu_chatbot_query(
    "Tell me about stories in the Science Fiction genre"
)

print("Method:", genre_result["method"])
print("LLM used:", genre_result["llm_used"])
print(
    "Latency:",
    f"{genre_result['retrieval_ms']:.2f} ms",
)
print(genre_result["answer"])

Method: genre_lookup
LLM used: False
Latency: 0.69 ms
- ID 457580: The Chronicles of the Cosmic Rift
- ID 377220: The Chronicles of the Quantum Voyagers
- ID 873393: The Quantum Chronicles: The Quest for the Lost Cosmos
- ID 979918: The Interstellar Odyssey
- ID 499422: Beyond the Neural Horizon
- ID 968752: The Time Travelers of Aetheron
- ID 331172: Galactic Odyssey: The Battle for the Cosmos
- ID 661488: The Chronicles of Elysium: Rise of the Galactic Defenders
- ID 788500: The Galactic Odyssey: A Tale of Two Planets
- ID 734908: Cosmic Leap into the Unknown


In [51]:
final_rss_gib = (
    process.memory_info().rss / 1024**3
)

print(f"Final RAG RSS: {final_rss_gib:.2f} GiB")

Final RAG RSS: 2.55 GiB


## saving results to drive


In [52]:
final_result = {
    "experiment": (
        "task1_cpu_qwen2.5_1.5b_q4_k_m"
    ),
    "hardware": {
        "cpu": platform.processor()
        or "Intel Xeon @ 2.20GHz",
        "logical_cpus": psutil.cpu_count(
            logical=True
        ),
        "physical_cpus": psutil.cpu_count(
            logical=False
        ),
        "threads_used": 2,
    },
    "model": {
        "name": "Qwen2.5-1.5B-Instruct",
        "format": "GGUF",
        "quantization": "Q4_K_M",
        "file_size_mib": (
            os.path.getsize(SMALL_Q4_GGUF)
            / 1024**2
        ),
        "context_size": 2048,
        "max_output_tokens": 96,
        "load_time_s": small_model_load_s,
    },
    "llama_bench": {
        "prompt_processing_pp128_tok_s": 27.95,
        "generation_tg128_tok_s": 11.43,
        "threads": 2,
    },
    "retrieval": {
        "method": (
            "chunk dense + BM25 + RRF"
        ),
        "context_policy": (
            "best query-relevant chunk, "
            "maximum 350 words"
        ),
    },
    "memory": {
        "model_only_session_rss_gib": (
            rss_after_model_load_gib
        ),
        "full_rag_rss_gib": final_rss_gib,
    },
    "queries": deployment_results,
}

FINAL_RESULTS_PATH = (
    f"{BENCHMARK_DIR}/"
    "task1_qwen15_cpu_final.json"
)

with open(FINAL_RESULTS_PATH, "w") as file:
    json.dump(
        final_result,
        file,
        indent=2,
        ensure_ascii=False,
    )

print("Saved:", FINAL_RESULTS_PATH)

Saved: /content/drive/MyDrive/task3_cpu_deployment/benchmarks/task1_qwen15_cpu_final.json


In [54]:
import time
import json
import numpy as np
from datetime import datetime, timezone

REPEATED_BENCHMARK_PATH = (
    f"{BENCHMARK_DIR}/"
    "task1_qwen15_repeated_latency.json"
)

latency_test_cases = [
    {
        "case_id": "id_detail",
        "query": (
            "What supernatural force threatens "
            "the family in story ID 523790?"
        ),
    },
    {
        "case_id": "semantic_theme",
        "query": (
            "Find me a story about adventure "
            "and friendship."
        ),
    },
    {
        "case_id": "rare_named_detail",
        "query": (
            "Which story involves the SS Excelsior "
            "and an emergency cloaking device?"
        ),
    },
]

N_RUNS_PER_QUERY = 3
MAX_OUTPUT_TOKENS = 96

latency_records = []

print(
    f"Running {len(latency_test_cases)} queries "
    f"x {N_RUNS_PER_QUERY} runs"
)

Running 3 queries x 3 runs


In [55]:
benchmark_started = time.perf_counter()

for run_number in range(1, N_RUNS_PER_QUERY + 1):
    print(f"\n{'#' * 90}")
    print(f"BENCHMARK ROUND {run_number}")
    print(f"{'#' * 90}")

    # Interleaving the query types reduces the chance that one query
    # receives systematically different thermal/runtime conditions.
    for case in latency_test_cases:
        query = case["query"]

        print("\nCase:", case["case_id"])
        print("Query:", query)

        result = cpu_chatbot_query(
            query,
            max_tokens=MAX_OUTPUT_TOKENS,
        )

        if not result.get("llm_used"):
            print("Skipped: this route did not use the LLM")
            continue

        record = {
            "case_id": case["case_id"],
            "query": query,
            "run": run_number,
            "method": result["method"],
            "retrieval_ms": float(
                result["retrieval_ms"]
            ),
            "prompt_tokens": int(
                result["prompt_tokens"]
            ),
            "completion_tokens": int(
                result["completion_tokens"]
            ),
            "ttft_s": float(
                result["ttft_s"]
            ),
            "generation_total_s": float(
                result["generation_total_s"]
            ),
            "end_to_end_s": float(
                result["end_to_end_s"]
            ),
            "sources": result["sources"],
            "answer": result["answer"],
        }

        latency_records.append(record)

        print(
            f"retrieval={record['retrieval_ms']:.1f} ms | "
            f"prompt={record['prompt_tokens']} tokens | "
            f"TTFT={record['ttft_s']:.2f}s | "
            f"total={record['end_to_end_s']:.2f}s"
        )

benchmark_wall_time_s = (
    time.perf_counter() - benchmark_started
)

print(
    f"\nRepeated benchmark completed in "
    f"{benchmark_wall_time_s:.1f}s"
)
print("Successful generated runs:", len(latency_records))


##########################################################################################
BENCHMARK ROUND 1
##########################################################################################

Case: id_detail
Query: What supernatural force threatens the family in story ID 523790?
retrieval=103.4 ms | prompt=611 tokens | TTFT=23.16s | total=24.95s

Case: semantic_theme
Query: Find me a story about adventure and friendship.
retrieval=66.4 ms | prompt=426 tokens | TTFT=16.75s | total=17.68s

Case: rare_named_detail
Query: Which story involves the SS Excelsior and an emergency cloaking device?
retrieval=68.2 ms | prompt=627 tokens | TTFT=22.59s | total=26.19s

##########################################################################################
BENCHMARK ROUND 2
##########################################################################################

Case: id_detail
Query: What supernatural force threatens the family in story ID 523790?
retrieval=38.6 ms | prompt=611 tokens

In [56]:
def latency_summary(records):
    retrieval_values = np.array(
        [item["retrieval_ms"] for item in records],
        dtype=float,
    )

    ttft_values = np.array(
        [item["ttft_s"] for item in records],
        dtype=float,
    )

    total_values = np.array(
        [item["end_to_end_s"] for item in records],
        dtype=float,
    )

    return {
        "n": int(len(records)),
        "retrieval_median_ms": float(
            np.median(retrieval_values)
        ),
        "retrieval_p95_ms": float(
            np.percentile(retrieval_values, 95)
        ),
        "ttft_mean_s": float(
            np.mean(ttft_values)
        ),
        "ttft_median_s": float(
            np.median(ttft_values)
        ),
        "ttft_p95_s": float(
            np.percentile(ttft_values, 95)
        ),
        "end_to_end_mean_s": float(
            np.mean(total_values)
        ),
        "end_to_end_median_s": float(
            np.median(total_values)
        ),
        "end_to_end_p95_s": float(
            np.percentile(total_values, 95)
        ),
        "end_to_end_min_s": float(
            np.min(total_values)
        ),
        "end_to_end_max_s": float(
            np.max(total_values)
        ),
    }


latency_by_case = {}

for case in latency_test_cases:
    case_records = [
        item
        for item in latency_records
        if item["case_id"] == case["case_id"]
    ]

    latency_by_case[case["case_id"]] = (
        latency_summary(case_records)
    )

overall_latency = latency_summary(
    latency_records
)

In [57]:
print("=== PER-QUERY LATENCY ===")

for case_id, summary in latency_by_case.items():
    print(f"\n{case_id} (n={summary['n']})")
    print(
        f"  Retrieval median: "
        f"{summary['retrieval_median_ms']:.1f} ms"
    )
    print(
        f"  TTFT median: "
        f"{summary['ttft_median_s']:.2f}s"
    )
    print(
        f"  TTFT P95: "
        f"{summary['ttft_p95_s']:.2f}s"
    )
    print(
        f"  Total median: "
        f"{summary['end_to_end_median_s']:.2f}s"
    )
    print(
        f"  Total P95: "
        f"{summary['end_to_end_p95_s']:.2f}s"
    )

print("\n=== OVERALL GENERATED LATENCY ===")
print("Runs:", overall_latency["n"])
print(
    f"Retrieval median: "
    f"{overall_latency['retrieval_median_ms']:.1f} ms"
)
print(
    f"Retrieval P95: "
    f"{overall_latency['retrieval_p95_ms']:.1f} ms"
)
print(
    f"TTFT mean: "
    f"{overall_latency['ttft_mean_s']:.2f}s"
)
print(
    f"TTFT median: "
    f"{overall_latency['ttft_median_s']:.2f}s"
)
print(
    f"TTFT P95: "
    f"{overall_latency['ttft_p95_s']:.2f}s"
)
print(
    f"End-to-end mean: "
    f"{overall_latency['end_to_end_mean_s']:.2f}s"
)
print(
    f"End-to-end median: "
    f"{overall_latency['end_to_end_median_s']:.2f}s"
)
print(
    f"End-to-end P95: "
    f"{overall_latency['end_to_end_p95_s']:.2f}s"
)
print(
    f"Range: "
    f"{overall_latency['end_to_end_min_s']:.2f}s–"
    f"{overall_latency['end_to_end_max_s']:.2f}s"
)

=== PER-QUERY LATENCY ===

id_detail (n=3)
  Retrieval median: 49.7 ms
  TTFT median: 21.31s
  TTFT P95: 22.97s
  Total median: 23.25s
  Total P95: 24.78s

semantic_theme (n=3)
  Retrieval median: 52.3 ms
  TTFT median: 16.02s
  TTFT P95: 16.68s
  Total median: 17.29s
  Total P95: 17.64s

rare_named_detail (n=3)
  Retrieval median: 54.6 ms
  TTFT median: 22.59s
  TTFT P95: 23.19s
  Total median: 26.19s
  Total P95: 26.71s

=== OVERALL GENERATED LATENCY ===
Runs: 9
Retrieval median: 53.8 ms
Retrieval P95: 89.4 ms
TTFT mean: 20.24s
TTFT median: 21.31s
TTFT P95: 23.22s
End-to-end mean: 22.35s
End-to-end median: 23.25s
End-to-end P95: 26.54s
Range: 16.70s–26.77s
